[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module4/02-pandas.ipynb)

# Pandas: DataFrames, GroupBy, Merges, and Pivot Tables

**Module 4 — Data Science & Visualization** | Estimated time: 30 minutes

## Learning Objectives

By the end of this notebook you will be able to:
- Create DataFrames and Series from multiple sources
- Select data with `loc`, `iloc`, boolean indexing, and `.query()`
- Transform data with `.assign()`, `.pipe()`, and method chaining
- Aggregate data using `groupby` with `agg`, `transform`, and `apply`
- Combine DataFrames with `merge` and `concat`
- Build pivot tables with margins

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f'Pandas version: {pd.__version__}')
print(f'NumPy  version: {np.__version__}')

# Reproducibility
rng = np.random.default_rng(42)

## 1. Creating DataFrames and Series

A **Series** is a 1D labeled array. A **DataFrame** is a 2D table where each column is a Series sharing the same index.

In [ ]:
# Series
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'], name='values')
print('Series:')
print(s)
print('s.index:', s.index.tolist())
print('s.dtype:', s.dtype)

# DataFrame from dict
df_dict = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'Dave'],
    'age':  [25, 30, 35, 28],
    'score': [88.5, 92.0, 79.3, 85.1]
})
print('\nDataFrame from dict:')
print(df_dict)

# DataFrame from list of dicts
records = [{'x': 1, 'y': 2}, {'x': 3, 'y': 4}, {'x': 5, 'y': 6}]
print('\nFrom list of dicts:')
print(pd.DataFrame(records))

# DataFrame from NumPy array
print('\nFrom NumPy array:')
print(pd.DataFrame(rng.integers(0, 100, (3, 4)), columns=['A', 'B', 'C', 'D']))

## 2. Indexing: loc, iloc, boolean, and query

Pandas offers multiple selection mechanisms. `loc` is label-based; `iloc` is integer-position-based. Boolean masks and `.query()` provide flexible filtering.

In [ ]:
df = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Doohickey', 'Thingamajig', 'Gizmo'],
    'category': ['A', 'B', 'A', 'C', 'B'],
    'price': [9.99, 24.99, 4.99, 49.99, 14.99],
    'stock': [150, 80, 300, 25, 120]
}, index=['p1', 'p2', 'p3', 'p4', 'p5'])

print('Full DataFrame:')
print(df)

# loc — label-based
print('\ndf.loc["p2"]:'); print(df.loc['p2'])
print('\ndf.loc[["p1","p3"], ["product","price"]]:'); print(df.loc[['p1','p3'], ['product','price']])

# iloc — position-based
print('\ndf.iloc[1:3, 0:3]:'); print(df.iloc[1:3, 0:3])

# Boolean indexing
print('\nProducts with price < 15:')
print(df[df['price'] < 15])

# .at and .iat for single values (fast)
print('\ndf.at["p4", "price"]:', df.at['p4', 'price'])

# .query() — SQL-like string filtering
print('\ndf.query("category == \'A\' and stock > 100"):')
print(df.query("category == 'A' and stock > 100"))

## 3. Transformations: assign, pipe, and Method Chaining

Method chaining keeps transformations readable. `.assign()` adds columns without mutating the original. `.pipe()` applies arbitrary functions to the whole DataFrame — great for custom steps.

In [ ]:
def add_value_score(frame):
    """Custom pipeline step: stock * price / 1000."""
    return frame.assign(value_score=(frame['stock'] * frame['price'] / 1000).round(2))

result = (
    df
    .assign(price_discounted=lambda x: (x['price'] * 0.9).round(2))
    .assign(is_low_stock=lambda x: x['stock'] < 100)
    .pipe(add_value_score)
    .sort_values('value_score', ascending=False)
)

print('Transformed DataFrame:')
print(result)
print('\nOriginal df is unchanged:')
print(df.head(2))

## 4. GroupBy Operations

`groupby` splits the DataFrame into groups, applies a function, and combines the results (Split-Apply-Combine). `agg` applies one or more aggregation functions; `transform` returns a same-size result; `apply` is the most flexible but slowest.

In [ ]:
# Build a larger dataset: 100 simulated sales records
np.random.seed(7)
n = 100
regions    = rng.choice(['North', 'South', 'East', 'West'], n)
products   = rng.choice(['Widget', 'Gadget', 'Gizmo'], n)
quantities = rng.integers(1, 50, n)
unit_price = {'Widget': 9.99, 'Gadget': 24.99, 'Gizmo': 14.99}
prices     = np.array([unit_price[p] for p in products])
discounts  = rng.uniform(0, 0.2, n).round(2)
revenue    = (quantities * prices * (1 - discounts)).round(2)

sales = pd.DataFrame({
    'region': regions,
    'product': products,
    'quantity': quantities,
    'unit_price': prices,
    'discount': discounts,
    'revenue': revenue
})

print('Sales dataset sample:')
print(sales.head())
print(f'Shape: {sales.shape}')

# Basic groupby + agg
print('\n--- Revenue by Region ---')
print(sales.groupby('region')['revenue'].agg(['sum', 'mean', 'count']).round(2))

# Multiple aggregations on multiple columns
print('\n--- Product stats ---')
print(sales.groupby('product').agg(
    total_revenue=('revenue', 'sum'),
    avg_qty=('quantity', 'mean'),
    avg_discount=('discount', 'mean'),
    n_orders=('revenue', 'count')
).round(2))

# transform: add group-level mean back to each row
sales['region_avg_rev'] = sales.groupby('region')['revenue'].transform('mean').round(2)
sales['above_region_avg'] = sales['revenue'] > sales['region_avg_rev']
print('\nSample with region benchmark:')
print(sales[['region', 'product', 'revenue', 'region_avg_rev', 'above_region_avg']].head(8))

## 5. Merging and Concatenating DataFrames

`pd.merge` performs SQL-style joins. `pd.concat` stacks DataFrames along rows or columns.

In [ ]:
# Two related tables
product_info = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Gizmo', 'Doohickey'],
    'category': ['Basic', 'Premium', 'Standard', 'Basic'],
    'supplier': ['SupplierA', 'SupplierB', 'SupplierA', 'SupplierC']
})

region_targets = pd.DataFrame({
    'region': ['North', 'South', 'East', 'West'],
    'quarterly_target': [15000, 12000, 18000, 14000]
})

# Inner join: only rows with matching keys in both tables
df_merged = pd.merge(sales, product_info, on='product', how='inner')
print('After merging product_info (inner):')
print(df_merged[['region', 'product', 'category', 'supplier', 'revenue']].head(5))

# Left join: all rows from sales, matched rows from region_targets
df_full = pd.merge(df_merged, region_targets, on='region', how='left')
df_full['pct_of_target'] = (df_full['revenue'] / df_full['quarterly_target'] * 100).round(2)
print('\nWith region targets:')
print(df_full[['region', 'product', 'revenue', 'quarterly_target', 'pct_of_target']].head(5))

# concat: stack two DataFrames row-wise
half1 = sales.iloc[:50].copy()
half2 = sales.iloc[50:].copy()
rejoined = pd.concat([half1, half2], ignore_index=True)
print(f'\npd.concat result shape: {rejoined.shape} (same as original {sales.shape})')

## 6. Pivot Tables

Pivot tables reshape data for cross-tabulation analysis. The `margins=True` parameter adds row and column totals.

In [ ]:
pivot = pd.pivot_table(
    sales,
    values='revenue',
    index='region',
    columns='product',
    aggfunc='sum',
    margins=True,
    margins_name='TOTAL'
).round(2)

print('Pivot Table — Revenue by Region x Product:')
print(pivot)

# Visualize the pivot (excluding totals row/col)
pivot_data = pivot.iloc[:-1, :-1]

fig, ax = plt.subplots(figsize=(8, 4))
pivot_data.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_title('Revenue by Region and Product')
ax.set_xlabel('Region')
ax.set_ylabel('Revenue ($)')
ax.legend(title='Product')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

# Quantity pivot with mean discount
print('\nMean Discount by Region x Product:')
print(pd.pivot_table(
    sales,
    values='discount',
    index='region',
    columns='product',
    aggfunc='mean',
    margins=True
).round(3))

## 7. Practical: Full Sales Analysis Pipeline

In [ ]:
summary = (
    sales
    .assign(revenue_per_unit=lambda x: (x['revenue'] / x['quantity']).round(2))
    .groupby(['region', 'product'])
    .agg(
        orders=('revenue', 'count'),
        total_revenue=('revenue', 'sum'),
        total_qty=('quantity', 'sum'),
        avg_discount=('discount', 'mean'),
        avg_rev_per_unit=('revenue_per_unit', 'mean')
    )
    .round(2)
    .sort_values('total_revenue', ascending=False)
)

print('Complete Sales Summary (top 10 region-product combos):')
print(summary.head(10).to_string())

# Top performing region overall
top_region = sales.groupby('region')['revenue'].sum().idxmax()
top_rev = sales.groupby('region')['revenue'].sum().max()
print(f'\nTop region: {top_region} with ${top_rev:,.2f} in revenue')

# Best-selling product by quantity
top_product = sales.groupby('product')['quantity'].sum().idxmax()
print(f'Most units sold: {top_product}')

## Practice Exercises

**Exercise 1 — GroupBy and Transform**
Using the `sales` DataFrame, add a new column `revenue_rank` that ranks each sale within its region (rank 1 = highest revenue in that region). Use `groupby` + `transform` with a lambda. How many rank-1 sales are there per region?

**Exercise 2 — Multi-level Merge**
Create a new DataFrame `supplier_costs` with columns `supplier` and `cost_per_unit` (make up values). Merge it with `df_merged` and calculate `profit_margin = (revenue - quantity * cost_per_unit) / revenue` for each order.

**Exercise 3 — Pivot Insights**
Create a pivot table showing the **total quantity** sold by region (rows) and product (columns). Then write a one-line expression using `.idxmax(axis=1)` to find the best-selling product in each region.